### Transformer

<img src="Architecture.png" width="500px">

#### Imports

In [1]:
import torch
import torch.nn as nn
import math

#### Input embedding layer

<img src="diagrams/InputLayer.png" width="1000px">

<img src="diagrams/InputEmbedding.png" width="1000px">

In [ ]:
class InputEmbedding(nn.Module):
     def __init__(self, vocabulary_size: int, dimensionE: int) -> None:
          super().__init__()

          self.dimensionE = dimensionE
          self.embedding = nn.Embedding(vocabulary_size, dimensionE)

     def forward(self, tokenIDs):
          embeddingStrength = math.sqrt(self.dimensionE)
          return self.embedding(tokenIDs) * embeddingStrength

In [3]:
tokenIDs = torch.tensor([2, 43, 21])
ie = InputEmbedding(vocabulary_size=200, dimensionE=6)(tokenIDs)
ie

tensor([[ 1.4072,  0.3093, -3.7628,  2.6932,  3.3901, -4.2940],
        [ 0.7901,  2.1676, -0.0258, -0.0317, -2.1810, -3.5762],
        [-1.5940, -2.8877, -5.1279,  1.4628, -1.7443,  2.7067]],
       grad_fn=<MulBackward0>)

#### Positional encoding

<img src="diagrams/PositionalEncoding.png" width="1000px">

In [4]:
class PositionalEncoding(nn.Module):
     def __init__(self, sequenceLength: int ,dimensionE: int, dropout: float) -> None:
          super().__init__()

          self.sequenceLength = sequenceLength
          self.dimensionE = dimensionE
          self.dropout = nn.Dropout(dropout)

          peMatrix = torch.zeros(sequenceLength, dimensionE)
          posVector = torch.arange(0, sequenceLength, dtype=torch.float).unsqueeze(1)
          frequencyCreator = torch.exp(torch.arange(0, dimensionE, 2).float() * (-math.log(10000.0) / dimensionE))

          peMatrix[:, 0::2] = torch.sin(posVector * frequencyCreator)
          peMatrix[:, 1::2] = torch.cos(posVector * frequencyCreator)

          peMatrix = peMatrix.unsqueeze(0)    # (1, sequenceLength, dimensionE)

          self.register_buffer("peMatrix", peMatrix)

     def forward(self, inputEmbedding):
          inputEmbedding = inputEmbedding + self.peMatrix[: , :inputEmbedding.shape[1], :].requires_grad_(False)
          return self.dropout(inputEmbedding)

In [5]:
PositionalEncoding(sequenceLength=3, dimensionE=6, dropout=0.1)(ie.unsqueeze(0))

tensor([[[ 1.5635,  1.4548, -4.1809,  4.1036,  3.7668, -0.0000],
         [ 1.8128,  0.0000,  0.0229,  1.0747, -2.4210, -2.8625],
         [-0.7608, -3.6709, -5.5946,  0.0000, -1.9334,  4.1185]]],
       grad_fn=<MulBackward0>)

#### Multi Head Attention

<img src="diagrams/MultiHeadAttention1.png" width="1000px"> 

<img src="diagrams/MultiHeadAttention2.png" width="1000px">

In [6]:
class MultiHeadAttentionBlock(nn.Module):
     def __init__(self, heads: int, dropout: float, dimensionE: int) -> None:
          super().__init__()

          self.dimensionE = dimensionE
          self.dropout = nn.Dropout(dropout)
          self.heads = heads
          assert dimensionE % heads == 0, "Embedding vector must be equally divided among attention heads"
          self.sizeHeadVector = dimensionE // heads
          self.wQ = nn.Linear(dimensionE, dimensionE, bias=False)
          self.wK = nn.Linear(dimensionE, dimensionE, bias=False)
          self.wV = nn.Linear(dimensionE, dimensionE, bias=False)
          self.wO = nn.Linear(dimensionE, dimensionE, bias=False)

     @staticmethod
     def attention(query, key, value, mask, dropout: nn.Dropout):
          dimensionKey = key.shape[-1]
          attentionValues = query @ key.transpose(-2, -1)
          attentionValues /= math.sqrt(dimensionKey)
          if mask is not None:
               attentionValues = attentionValues.masked_fill(mask==0, -1e9)
          attentionValues = attentionValues.softmax(dim=-1)
          if dropout is not None:
               attentionValues = dropout(attentionValues)
          return attentionValues @ value, attentionValues
     
     def forward(self, q, k, v, mask):
          # (batch, sequenceLength, embedding vector size)
          query = self.wQ(q)         
          key = self.wK(k)
          value = self.wV(v)

          #  (batch size, sequenceLength, heads, embedding vector size)     to     (batch size, heads, sequenceLength, embedding vector size)
          query = query.view(query.shape[0], query.shape[1], self.heads, self.sizeHeadVector)      
          query = query.transpose(1, 2)            

          key = key.view(key.shape[0], key.shape[1], self.heads, self.sizeHeadVector)      
          key = key.transpose(1, 2)

          value = value.view(value.shape[0], value.shape[1], self.heads, self.sizeHeadVector)      
          value = value.transpose(1, 2)      

          attentionQKV, attentionValues = MultiHeadAttentionBlock.attention(
               query = query,
               key = key,
               value = value,
               mask = mask,
               dropout = self.dropout
          )

          attentionQKV = attentionQKV.transpose(1,2)
          attentionQKV = attentionQKV.contiguous()
          attentionQKV = attentionQKV.view(attentionQKV.shape[0], -1, self.heads * self.sizeHeadVector)      # batch, sequenceLength, dimensionE

          return self.wO(attentionQKV)

#### Layer Normalisation

In [7]:
class LayerNormalisation(nn.Module):
     def __init__(self, dimensionE: int, microValue: float = 10**-7) -> None:
          super().__init__()

          self.dimensionE = dimensionE
          self.microValue = microValue
          self.gamma = nn.Parameter(torch.ones(dimensionE))
          self.beta = nn.Parameter(torch.zeros(dimensionE))

     def forward(self, input):
          mean = input.mean(dim=-1, keepdim=True)
          std = input.std(dim=-1, keepdim=True)
          normalisedInput = (input - mean) / (std + self.microValue)

          return self.gamma * normalisedInput + self.beta

#### Feed forward block

In [8]:
class FeedForwardBlock(nn.Module):
     def __init__(self, dimensionE: int, neurons: int, dropout: float) -> None:
          super().__init__()

          self.dimensionE = dimensionE
          self.neurons = neurons
          self.layer1 = nn.Linear(dimensionE, neurons)
          self.dropout = nn.Dropout(dropout)
          self.layer2 = nn.Linear(neurons, dimensionE)

     def forward(self, input):
          return self.layer2(self.dropout(torch.relu((self.layer1(input)))))

#### Residual Connection

In [10]:
class ResidualConnection(nn.Module):
     def __init__(self, dimensionE: int, dropout: float) -> None:
          super().__init__()
          self.dropout = nn.Dropout(dropout)
          self.normalise = LayerNormalisation(dimensionE)

     def forward(self, input, sublayer):
          connection = self.dropout(sublayer(self.normalise(input))) + input
          return connection